In [ ]:
import numpy as np
from IPython.display import display

from pyquiver import KIE_Calculation, System, Config, Isotopologue, batch, parsers
from pyquiver.constants import DEFAULT_MASSES, REPLACEMENTS
from pyquiver.kie import calculate_rpfr, partition_components

## Supported input files

PyQuiver reads Cartesian Hessians from several electronic-structure programs. Pass the matching `style` (the first name; the rest are aliases):

| Program | `style` | files |
|---------|---------|-------|
| Gaussian | `gaussian` (`g16`, `g09`) | `.out` |
| ORCA | `orca` | `.hess` |
| PyQuiver native | `pyquiver` (`native`, `qin`) | `.qin` |

In [ ]:
parsers.supported_styles()

## A simple calculation

Build the configuration in Python with `Config.from_dict`, then run it on a ground-state and a transition-state file. Each substitution is `(ground_state_atom, transition_state_atom, replacement)` with 1-indexed atoms; `replacement` is a `weights.dat` label such as `"13C"`, `"2D"`, or `"17O"` (one isotopologue can make several). (You can also load a `.config` file with `Config(path)`.)

In [ ]:
config = Config.from_dict(
    isotopologues={
        "C1":  [(1, 1, "13C")],
        "C4":  [(4, 4, "13C")],
        "O3":  [(3, 3, "17O")],
        "H/D": [(7, 7, "2D"), (8, 8, "2D")],
    },
    temperature=393, scaling=0.9614, imag_threshold=50,
    reference_isotopologue="none",
)

calc = KIE_Calculation(config, "../tutorial/gaussian/claisen_gs.out", "../tutorial/gaussian/claisen_ts.out",
                       style="gaussian")
print(calc)

## Results as data

Look up a single isotopologue by name, or get the whole set as a table.

In [ ]:
c4 = calc.results["C4"]
print(f"C4 -- uncorrected: {c4.uncorrected:.4f}, Wigner: {c4.wigner:.4f}, inverted parabola: {c4.inverted_parabola:.4f}")

display(calc.to_dataframe())

## Reusing parsed files

A `System` holds a parsed geometry and Hessian. Build the ground and transition states once and reuse them -- for example to scan the temperature -- so each file is read only once.

In [ ]:
gs = System("../tutorial/gaussian/claisen_gs.out")
ts = System("../tutorial/gaussian/claisen_ts.out")

for T in (298, 350, 393):
    cfg = Config.from_dict(isotopologues={"C4": [(4, 4, "13C")]},
                           temperature=T, scaling=0.9614, imag_threshold=50)
    print(f"{T} K: C4 KIE = {KIE_Calculation(cfg, gs, ts).results['C4'].uncorrected:.4f}")

## Unusual isotopes

A `replacement` is normally a standard isotope label, but a bare number is used directly as a mass (amu) -- so non-standard isotopes need no `weights.dat` entry. Here we place a (fictitious) mass-5000 nucleus at atom 4:

In [ ]:
cfg = Config.from_dict(isotopologues={"heavyC4": [(4, 4, 5000.0)]},
                       temperature=393, scaling=0.9614, imag_threshold=50)
print(KIE_Calculation(cfg, gs, ts).results["heavyC4"])

## ORCA input

ORCA `.hess` files are read with `style="orca"`; nothing else changes.

In [ ]:
orca = KIE_Calculation(config, "../tutorial/orca/claisen_gs_freq.hess",
                       "../tutorial/orca/claisen_ts_freq.hess", style="orca")
display(orca.to_dataframe())

## Many files at once: `batch`

`batch` runs one configuration over many ground-state/transition-state pairs and collects the results into one table. Build the pairs from your own file globbing -- no naming convention is assumed. Pairs may be a `{label: (gs, ts)}` mapping, `(label, gs, ts)` triples, or bare `(gs, ts)` tuples (label inferred from the filename).

In [ ]:
results = batch(config, {
    "run1": ("../tutorial/gaussian/claisen_gs.out", "../tutorial/gaussian/claisen_ts.out"),
    "run2": ("../tutorial/gaussian/claisen_gs.out", "../tutorial/gaussian/claisen_ts.out"),
})

display(results.to_dataframe())
results["run1"]   # the KIE_Calculation for that pair, for drilling down

## Tunnelling corrections

Every KIE is reported three ways -- **uncorrected**, **Wigner**, and **Bell** (inverted parabola). Wigner and Bell need only the transition state's imaginary frequency; the **Skodje-Truhlar** correction also needs the barrier *height* (the reactant, product, and TS energies) and can be more appropriate when the imaginary frequency is very large.

For an imaginary mode of magnitude $\tilde\nu^\ddagger$, write $u = \dfrac{h c\,\tilde\nu^\ddagger}{k_B T}$. The single-mode transmission coefficients are

$$\kappa_{\text{Wigner}} = 1 + \frac{u^2}{24}, \qquad \kappa_{\text{Bell}} = \frac{u/2}{\sin(u/2)}.$$

Skodje-Truhlar additionally uses $\alpha = \dfrac{2\pi}{h\,\nu^\ddagger}$, $\beta = \dfrac{1}{k_B T}$, and the barrier $V = V_{\text{TS}} - \max(V_{\text{reactant}}, V_{\text{product}})$:

$$\kappa = \frac{\beta\pi/\alpha}{\sin(\beta\pi/\alpha)} - \frac{\beta}{\alpha-\beta}\,e^{(\beta-\alpha)V} \quad (\alpha > \beta),$$

$$\kappa = \frac{\beta}{\beta-\alpha}\left(e^{(\beta-\alpha)V} - 1\right) \quad (\alpha < \beta).$$

In every case the KIE is multiplied by $\kappa_{\text{light}}/\kappa_{\text{heavy}}$. The uncorrected, Wigner, and Bell values are in `to_dataframe()`; Skodje-Truhlar is computed on demand because it needs electronic energies, read here with [`cctk`](https://github.com/ekwan/cctk) (`pip install cctk`).

In [ ]:
import cctk

def energy(path):
    ensemble = cctk.GaussianFile.read_file(path).ensemble
    return ensemble.get_properties_dict(ensemble.molecules[-1])["energy"]

E_reactant = energy("../tutorial/gaussian/claisen_gs.out")
E_ts = energy("../tutorial/gaussian/claisen_ts.out")

# energies in hartree; product taken equal to the reactant here
calc.skodje_truhlar(reactant_energy=E_reactant, product_energy=E_reactant, ts_energy=E_ts)

## Under the hood: reduced isotopic partition function ratios

A KIE is a ratio of reduced isotopic partition function ratios (RPFRs). You can compute these directly: build a light and a heavy `Isotopologue` and call `calculate_rpfr`.

In [ ]:
gs_masses = [DEFAULT_MASSES[z] for z in gs.atomic_numbers]
heavy_masses = list(gs_masses)
heavy_masses[0] = REPLACEMENTS["13C"]        # 13C at atom 1

light = Isotopologue("light", gs, gs_masses)
heavy = Isotopologue("heavy", gs, heavy_masses)

rpfr, imag_ratio, light_freqs, heavy_freqs = calculate_rpfr((light, heavy), 50.0, 0.9614, 393)
print(f"ground-state RPFR = {rpfr:.4f}")

The per-mode contributions are returned data, not a debug log. `partition_components` gives the product, excitation, and zero-point factors for every vibrational mode:

In [ ]:
import pandas as pd

components = partition_components(heavy_freqs, light_freqs, 393)
modes = pd.DataFrame(components, columns=["product", "excitation", "ZPE"])
modes["contribution"] = modes.prod(axis=1)
display(modes.head())